In [ ]:
# -------------------------------------------------------------
# Libraries and Imports
# -------------------------------------------------------------

# Paths and File Handling
import subprocess
from pathlib import Path
from jinja2 import Template
from typing import Optional
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional
import gmsh
import pyarrow as pa
import pyarrow.parquet as pq

# Numerical and Visualization Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pyvista as pv
import h5py

# Neural Network and Machine Learning Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.ndimage import label
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from scipy.stats import binned_statistic_2d
from concurrent.futures import ThreadPoolExecutor

# Additional utilities
import os
import time
import random
import copy
import warnings
warnings.filterwarnings("ignore", message="An output with one or more elements was resized")
from collections import defaultdict

In [ ]:
# -------------------------------------------------------------
# Torch Device Configuration
# -------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
def _classify_boundary_curves(fluid_surfaces, Lx, Ly, tol=1e-6):
    """Sort the outer boundary curves of the fluid domain into groups.
 
    A curve belongs to a side only if its ENTIRE bounding box lies on that side
    (e.g. inlet: xmin and xmax both ~ 0). Checking only xmin would wrongly
    classify the top/bottom edges that start at x = 0 as inlet.
    """
    groups = {"Inlet": [], "Outlet": [], "TopBottom": [], "Wall": []}
    seen = set()
    for dim, tag in gmsh.model.getBoundary([(2, s) for s in fluid_surfaces],
                                           combined=True, oriented=False):
        if tag in seen:
            continue
        seen.add(tag)
        xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
        if abs(xmin) < tol and abs(xmax) < tol:
            groups["Inlet"].append(tag)
        elif abs(xmin - Lx) < tol and abs(xmax - Lx) < tol:
            groups["Outlet"].append(tag)
        elif (abs(ymin) < tol and abs(ymax) < tol) or \
             (abs(ymin - Ly) < tol and abs(ymax - Ly) < tol):
            groups["TopBottom"].append(tag)
        else:
            groups["Wall"].append(tag)  # internal solid (fin) surfaces
    return groups

def generate_mesh(Lx, Ly, nx, ny, fin_coords, ID, verbose: bool = False,
                  structured: bool = True,
                  size_min=None, size_max=None, refine_dist=None,
                  refine_outer_walls: bool = False) -> str:
    """Write a .msh file for MOOSE using gmsh.
 
    structured=True  : original behaviour, nx-by-ny grid of constant-size quads.
    structured=False : unstructured, gmsh-controlled sizing with refinement near
                       the fin walls. Output is guaranteed to be all quads.
 
    Unstructured-only parameters (these are the FINAL element sizes):
        size_min           element size next to refined walls (default min(dx,dy)/2)
        size_max           element size far from walls         (default max(dx,dy))
        refine_dist        distance from the wall over which size grows from
                           size_min to size_max               (default max(Lx,Ly)/10)
        refine_outer_walls also refine along top/bottom walls  (default False)
 
    Physical groups: Fluid (2D); Inlet, Outlet, TopBottom, Wall (1D).
    "Wall" now contains only the internal fin surfaces.
    """
    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
        gmsh.model.add(f"hx_{ID}")
 
        dx = Lx / nx
        dy = Ly / ny
 
        # ---- Geometry: rectangle minus fins --------------------------------
        rect = gmsh.model.occ.addRectangle(0, 0, 0, Lx, Ly)
        gmsh.model.occ.synchronize()
 
        obstructions = []
        for polygon in fin_coords:
            pts = [gmsh.model.occ.addPoint(x, y, 0) for x, y in polygon]
            lines = [gmsh.model.occ.addLine(pts[i], pts[(i + 1) % len(pts)])
                     for i in range(len(pts))]
            cl = gmsh.model.occ.addCurveLoop(lines)
            obstructions.append(gmsh.model.occ.addPlaneSurface([cl]))
        gmsh.model.occ.synchronize()
 
        if obstructions:
            cut = gmsh.model.occ.cut([(2, rect)], [(2, s) for s in obstructions],
                                     removeObject=True, removeTool=True)
            gmsh.model.occ.synchronize()
            fluid_surfaces = [tag for dim, tag in cut[0] if dim == 2]
        else:
            fluid_surfaces = [rect]
 
        if not fluid_surfaces:
            raise RuntimeError("Boolean cut produced no fluid domain.")
 
        # ---- Structured only: split domain along the grid lines -------------
        if structured:
            v_lines = [gmsh.model.occ.addLine(
                gmsh.model.occ.addPoint(i * dx, 0, 0),
                gmsh.model.occ.addPoint(i * dx, Ly, 0)) for i in range(1, int(nx))]
            h_lines = [gmsh.model.occ.addLine(
                gmsh.model.occ.addPoint(0, j * dy, 0),
                gmsh.model.occ.addPoint(Lx, j * dy, 0)) for j in range(1, int(ny))]
            gmsh.model.occ.synchronize()
 
            frag, _ = gmsh.model.occ.fragment([(2, s) for s in fluid_surfaces],
                                              [(1, l) for l in v_lines + h_lines])
            gmsh.model.occ.synchronize()
            fluid_surfaces = [tag for dim, tag in frag if dim == 2]
            if not fluid_surfaces:
                raise RuntimeError("Fragmentation produced no fluid domain.")
 
        gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")
 
        # ---- Boundary groups -------------------------------------------------
        groups = _classify_boundary_curves(fluid_surfaces, Lx, Ly)
        for name, tags in groups.items():
            if tags:
                gmsh.model.addPhysicalGroup(1, tags, name=name)
 
        # ---- Meshing controls -------------------------------------------------
        gmsh.option.setNumber("Mesh.RecombineAll", 1)
        gmsh.option.setNumber("Mesh.ElementOrder", 2)
        gmsh.option.setNumber("Mesh.SecondOrderLinear", 1)
 
        if structured:
            for dim, tag in gmsh.model.getEntities(1):
                xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
                lx, ly = abs(xmax - xmin), abs(ymax - ymin)
                n_cells = max(1, round(lx / dx if lx > ly else ly / dy))
                gmsh.model.mesh.setTransfiniteCurve(tag, n_cells + 1)
            for tag in fluid_surfaces:
                try:
                    gmsh.model.mesh.setTransfiniteSurface(tag)
                except Exception:
                    pass
                gmsh.model.mesh.setRecombine(2, tag)
 
            gmsh.option.setNumber("Mesh.MeshSizeMin", dx)
            gmsh.option.setNumber("Mesh.MeshSizeMax", dy)
            gmsh.option.setNumber("Mesh.Algorithm", 5)
            gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 2)
        else:
            # Subdivision splits every element into 4 quads, which halves the
            # element size, so the sizing fields are doubled to compensate.
            sub = 2.0
            h_min = (size_min if size_min is not None else min(dx, dy) / 2) * sub
            h_max = (size_max if size_max is not None else max(dx, dy)) * sub
            dist = refine_dist if refine_dist is not None else max(Lx, Ly) / 10
 
            # Distance-based size field from the walls we want refined
            refine_curves = list(groups["Wall"])
            if refine_outer_walls:
                refine_curves += groups["TopBottom"]
 
            # Sizes must come only from the background field
            gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
            gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
            gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
            gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
            gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)
 
            if refine_curves:
                f_dist = gmsh.model.mesh.field.add("Distance")
                gmsh.model.mesh.field.setNumbers(f_dist, "CurvesList", refine_curves)
                gmsh.model.mesh.field.setNumber(f_dist, "Sampling", 100)
 
                f_thr = gmsh.model.mesh.field.add("Threshold")
                gmsh.model.mesh.field.setNumber(f_thr, "InField", f_dist)
                gmsh.model.mesh.field.setNumber(f_thr, "SizeMin", h_min)
                gmsh.model.mesh.field.setNumber(f_thr, "SizeMax", h_max)
                gmsh.model.mesh.field.setNumber(f_thr, "DistMin", 0)
                gmsh.model.mesh.field.setNumber(f_thr, "DistMax", dist)
                gmsh.model.mesh.field.setAsBackgroundMesh(f_thr)
            else:
                # Nothing to refine around: uniform size
                gmsh.option.setNumber("Mesh.MeshSizeMin", h_max)
 
            gmsh.option.setNumber("Mesh.Algorithm", 8)                 # Frontal-Delaunay for quads
            gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 3)    # Blossom full-quad
            gmsh.option.setNumber("Mesh.SubdivisionAlgorithm", 1)      # split everything into quads
 
        gmsh.model.mesh.generate(2)
 
        # ---- Safety check: only quads allowed --------------------------------
        elem_types, _, _ = gmsh.model.mesh.getElements(2)
        quad_types = {3, 10, 16}  # Quad4, Quad9, Quad8
        bad = [t for t in elem_types if t not in quad_types]
        if bad:
            names = [gmsh.model.mesh.getElementProperties(t)[0] for t in bad]
            raise RuntimeError(f"Mesh contains non-quad elements: {names}")
 
        mesh_filename = f"baseline-{ID}.msh"
        gmsh.write(mesh_filename)
        return mesh_filename
    finally:
        gmsh.finalize()

In [ ]:
def create_input_file(
    mesh_name: str,
    template_path: str | Path,
    output_dir: str | Path,
    fluid_properties: dict,
    boundary_conditions: dict,
) -> Path:
    """
    Render the Jinja2 MOOSE .i template for one mesh and write it to
    disk. Returns the path to the written .i file.

    boundary_conditions must contain U_in, T_in, T_wall, P_pin.
    Optional keys:
        U_profile_type : "uniform" (default) or "parabolic"
        domain_height  : channel height in m (required for "parabolic")

    U_in is the MEAN inlet velocity for both profile types.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    template_path = Path(template_path)

    case_id = Path(mesh_name).stem
    input_filename = f"{case_id}.i"

    # ---------------------------------------------------------
    # Inlet velocity profile
    # ---------------------------------------------------------
    U = boundary_conditions["U_in"]
    profile_type = boundary_conditions.get("U_profile_type", "uniform")

    if profile_type == "uniform":
        inlet_expression = f"{U:.10g}"

    elif profile_type == "parabolic":
        H = boundary_conditions.get("domain_height")
        if H is None or H <= 0:
            raise ValueError(
                "boundary_conditions['domain_height'] must be a positive "
                "number for a parabolic inlet profile."
            )
        # Fully developed plane Poiseuille flow, channel spans y = 0..H:
        #   u(y) = 6 * U_mean * y * (H - y) / H^2
        # Peak velocity is 1.5 * U_mean, and the mean over the height is U.
        coeff = 6.0 * U / H**2
        inlet_expression = f"{coeff:.10g}*y*({H:.10g}-y)"

    else:
        raise ValueError(
            f"Unknown U_profile_type '{profile_type}'. "
            "Use 'uniform' or 'parabolic'."
        )

    with open(template_path, "r") as f:
        template = Template(f.read())

    rendered = template.render(
        mesh_name=mesh_name,
        case_name=case_id,
        rho=fluid_properties["rho"],
        mu=fluid_properties["mu"],
        k=fluid_properties["k"],
        cp=fluid_properties["cp"],
        U_in=U,
        T_in=boundary_conditions["T_in"],
        T_wall=boundary_conditions["T_wall"],
        P_pin=boundary_conditions["P_pin"],
        inlet_expression=inlet_expression,
    )

    output_path = output_dir / input_filename
    with open(output_path, "w") as f:
        f.write(rendered)

    return output_path

def _run_moose(input_file: str, n_processors: int, verbose: bool = True) -> None:
    """Execute MOOSE via subprocess inside the 'moose' conda environment. Raises on failure."""
    cmd = [
        "conda", "run", "-n", "moose",
        "mpiexec", "-np", str(n_processors),
        "./cutthroat-opt", "-i", input_file,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"MOOSE simulation failed for {input_file} with exit code {result.returncode}.\n"
            f"Error output:\n{result.stderr}"
        )
    if verbose:
        print(result.stdout)

def _clean_input_file(input_file: str | Path) -> None:
    """Delete the temporary MOOSE .i input file after simulation."""
    try:
        os.remove(input_file)
    except OSError as e:
        print(f"Failed to delete input file {input_file}: {e}")

def read_moose_exodus(exodus_path: str | Path) -> dict[str, np.ndarray]:
    """Reads an Exodus II file -> {'coords', 'u', 'v', 'p', 'T'} at the last time step."""
    exodus_path = str(exodus_path)
    reader = pv.get_reader(exodus_path)
    reader.set_active_time_value(reader.time_values[-1])
    mb = reader.read()

    def _valid_blocks(block):
        out = []
        if isinstance(block, pv.MultiBlock):
            for i in range(block.n_blocks):
                out.extend(_valid_blocks(block[i]))
        elif block is not None and block.n_points > 0 and block.n_cells > 0:
            out.append(block)
        return out

    valid_blocks = _valid_blocks(mb)
    if not valid_blocks:
        raise RuntimeError(f"No non-empty blocks found in {exodus_path}.")

    mesh = pv.MultiBlock(valid_blocks).combine().cell_data_to_point_data()
    coords = np.array(mesh.points, dtype=np.float64)
    vel = np.array(mesh["vel_"], dtype=np.float64)
    return {
        "coords": coords,
        "u": vel[:, 0],
        "v": vel[:, 1],
        "p": np.array(mesh["p"], dtype=np.float64),
        "T": np.array(mesh["T"], dtype=np.float64),
    }

def display_programmatic_hx_diagnostics(results: dict, boundary_conditions: dict) -> None:
    """Display diagnostic plots for the programmatically generated heat exchanger."""
    # Example diagnostic plot (replace with actual implementation)
    pass

In [ ]:
def generate_programmatic_hx(
    domain_height: float,
    ny: int,
    nx: int,
    L_inlet: float,
    L_hx: float,
    L_outlet: float,
    n_fins: int | None = None,
    fin_thickness: int = 1,
    generate_mesh_file: bool = True,
    ID: str | None = None,
    verbose: bool = False,
    structured: bool = True,
    size_min: float | None = None,
    size_max: float | None = None,
    refine_dist: float | None = None,
    refine_outer_walls: bool = False
):
    """
    Generate a programmatic heat exchanger geometry and optionally
    generate its Gmsh mesh.

    Parameters
    ----------
    domain_height : float
        Total channel height.

    ny : int
        Number of grid cells in the y-direction.

    nx : int
        Number of grid cells in the x-direction.

    L_inlet : float
        Length of the inlet section.

    L_hx : float
        Length of the heat exchanger section.

    L_outlet : float
        Length of the outlet section.

    n_fins : int or None, optional
        Number of horizontal fins in the HX region.
        Use None for an empty channel.

    fin_thickness : int, optional
        Fin thickness in grid cells.

    generate_mesh_file : bool, optional
        If True, generate and save the Gmsh mesh.

    ID : str or None, optional
        Unique identifier for the mesh filename. If None, one is
        automatically generated.

    verbose : bool, optional
        If True, print Gmsh output.

    Returns
    -------
    str or list
        If generate_mesh_file=True, returns the mesh filename.
        Otherwise, returns the generated fin coordinates.
    """

    # ---------------------------------------------------------
    # Basic validation
    # ---------------------------------------------------------

    if nx <= 0 or ny <= 0:
        raise ValueError("nx and ny must be positive integers.")

    if domain_height <= 0:
        raise ValueError("domain_height must be positive.")

    if L_inlet < 0 or L_hx < 0 or L_outlet < 0:
        raise ValueError("Section lengths cannot be negative.")

    domain_length = L_inlet + L_hx + L_outlet

    if domain_length <= 0:
        raise ValueError("Total domain length must be positive.")

    if n_fins is not None:

        if n_fins <= 0:
            raise ValueError("n_fins must be positive or None.")

        if fin_thickness <= 0:
            raise ValueError("fin_thickness must be positive.")

        if n_fins * fin_thickness >= ny:
            raise ValueError(
                "Fins occupy the entire channel height. "
                "Increase ny or reduce n_fins/fin_thickness."
            )

    # ---------------------------------------------------------
    # Grid dimensions
    # ---------------------------------------------------------

    dx = domain_length / nx
    dy = domain_height / ny

    # ---------------------------------------------------------
    # Generate fin geometry
    # ---------------------------------------------------------

    fin_coords = []

    if n_fins is not None:

        # Number of fluid cells between fins.
        #
        # There are n_fins + 1 fluid regions:
        #
        #   fluid
        #   fin
        #   fluid
        #   fin
        #   ...
        #   fluid
        #
        available_fluid_cells = ny - n_fins * fin_thickness

        channel_height = available_fluid_cells / (n_fins + 1)

        # We want the geometry to remain aligned with the grid.
        channel_height = int(round(channel_height))

        if channel_height <= 0:
            raise ValueError(
                "No fluid cells remain between the fins."
            )

        # Check that rounding did not make the geometry impossible.
        total_cells_used = (
            n_fins * fin_thickness
            + (n_fins + 1) * channel_height
        )

        if total_cells_used > ny:
            raise ValueError(
                "Requested fin geometry cannot fit within the grid. "
                "Increase ny or reduce the fin dimensions."
            )

        # -----------------------------------------------------
        # Create fins
        # -----------------------------------------------------

        for i in range(n_fins):

            fin_bottom = (
                channel_height
                + i * (channel_height + fin_thickness)
            ) * dy

            fin_top = fin_bottom + fin_thickness * dy

            fin_left = L_inlet
            fin_right = L_inlet + L_hx

            fin_coords.append([
                (fin_left, fin_bottom),
                (fin_right, fin_bottom),
                (fin_right, fin_top),
                (fin_left, fin_top),
            ])

    # ---------------------------------------------------------
    # Return geometry without generating mesh
    # ---------------------------------------------------------

    if not generate_mesh_file:
        return fin_coords

    # ---------------------------------------------------------
    # Generate unique mesh ID
    # ---------------------------------------------------------

    if ID is None:
        ID = uuid.uuid4().hex[:8]

    # ---------------------------------------------------------
    # Generate Gmsh mesh
    # ---------------------------------------------------------

    mesh_filename = generate_mesh(
        Lx=domain_length,
        Ly=domain_height,
        nx=nx,
        ny=ny,
        fin_coords=fin_coords,
        ID=ID,
        verbose=verbose,
        structured=structured,
        size_min=size_min,
        size_max=size_max,
        refine_dist=refine_dist,
        refine_outer_walls=refine_outer_walls
    )

    return mesh_filename

def simulate_programmatic_hx(
    mesh_filename: str | Path,
    n_cores: int,
    fluid_properties: dict,
    boundary_conditions: dict,
    template_path: str | Path = "template.i",
    output_dir: str | Path = ".",
    results_dir: str | Path = "results",
    display_diagnostic: bool = True,
    verbose: bool = True,
):
    """
    Run a MOOSE simulation using a programmatically generated mesh.

    Parameters
    ----------
    mesh_filename : str or Path
        Path to the Gmsh .msh file.

    n_cores : int
        Number of MPI processes used by MOOSE.

    fluid_properties : dict
        Fluid properties. Must contain:
            rho
            mu
            k
            cp

    boundary_conditions : dict
        Boundary conditions. Must contain:
            U_in
            T_in
            T_wall
            P_pin

    template_path : str or Path
        Path to the Jinja2 MOOSE input template.

    output_dir : str or Path
        Directory where the temporary MOOSE .i file is created.

    results_dir : str or Path
        Directory where MOOSE writes the Exodus result.

    display_diagnostic : bool
        If True, display basic diagnostic plots after the simulation.

    verbose : bool
        If True, print MOOSE output.

    Returns
    -------
    dict
        Dictionary containing the simulation results.
    """

    mesh_filename = Path(mesh_filename)
    output_dir = Path(output_dir)
    results_dir = Path(results_dir)

    # ---------------------------------------------------------
    # Validate mesh
    # ---------------------------------------------------------

    if not mesh_filename.exists():
        raise FileNotFoundError(
            f"Mesh file not found: {mesh_filename}"
        )

    # ---------------------------------------------------------
    # Create MOOSE input file
    # ---------------------------------------------------------

    input_file = create_input_file(
        mesh_name=str(mesh_filename),
        template_path=template_path,
        output_dir=output_dir,
        fluid_properties=fluid_properties,
        boundary_conditions=boundary_conditions,
    )

    # MOOSE uses the mesh filename stem as the case name.
    case_name = mesh_filename.stem

    # This corresponds to:
    #
    # file_base = results/{{ case_name }}
    #
    # in template.i
    exodus_file = results_dir / f"{case_name}.e"

    try:

        # -----------------------------------------------------
        # Run MOOSE
        # -----------------------------------------------------

        print(f"\nRunning MOOSE simulation: {case_name}")
        print(f"Mesh: {mesh_filename}")
        print(f"Processors: {n_cores}")

        _run_moose(
            input_file=str(input_file),
            n_processors=n_cores,
            verbose=verbose,
        )

        # -----------------------------------------------------
        # Verify MOOSE output
        # -----------------------------------------------------

        if not exodus_file.exists():
            raise FileNotFoundError(
                f"MOOSE completed successfully, but the expected "
                f"Exodus file was not found:\n{exodus_file}"
            )

        print(f"\nMOOSE simulation completed.")
        print(f"Results: {exodus_file}")

        # -----------------------------------------------------
        # Read Exodus results
        # -----------------------------------------------------

        results = read_moose_exodus(exodus_file)

        # Keep some useful metadata with the results.
        results["mesh_filename"] = mesh_filename
        results["exodus_file"] = exodus_file
        results["case_name"] = case_name

        # -----------------------------------------------------
        # Diagnostics
        # -----------------------------------------------------

        if display_diagnostic:
            display_programmatic_hx_diagnostics(
                results,
                boundary_conditions=boundary_conditions,
            )

        return results

    finally:

        # -----------------------------------------------------
        # Remove temporary MOOSE input file
        # -----------------------------------------------------
        print(f"Cleaning up temporary input file: {input_file}")
        #_clean_input_file(input_file)

In [ ]:
# -------------------------------------------------------------
# Define the dimensions of the heat exchanger
# -------------------------------------------------------------

domain_height = 0.05  # meters

inlet_length = domain_height * 1
hx_length = domain_height * 1
outlet_length = domain_height * 1

domain_length = inlet_length + hx_length + outlet_length

# Initial mesh resolution
ny_initial = 300

# Number of additional refinement levels
n_refinements = 1


# -------------------------------------------------------------
# Fluid properties and boundary conditions
# -------------------------------------------------------------

fluid_properties = {
    "rho": 1000.0,       # density (kg/m^3)
    "mu": 0.001,         # dynamic viscosity (Pa*s)
    "k": 0.598,          # thermal conductivity (W/m*K)
    "cp": 4186.0,        # specific heat capacity (J/kg*K)
}

Re = 100
U = fluid_properties["mu"] * Re / (fluid_properties["rho"] * domain_height)
U_profile_type = "uniform"  # Options: "uniform", "parabolic"

boundary_conditions = {
    "U_in": U,                          # inlet velocity (m/s)
    "T_in": 273.15 + 45,                # inlet temperature (K)
    "T_wall": 353.15,                   # wall temperature (K)
    "P_pin": 1e5,                       # pressure pin (Pa)
    "U_profile_type": U_profile_type,   # inlet velocity profile type
    "domain_height": domain_height,     # required for parabolic profile
}

print(f"Re = {Re}, U = {U:.5f} m/s")


# -------------------------------------------------------------
# Generate mesh refinement cases
# -------------------------------------------------------------

mesh_tests = []

for i in range(n_refinements + 1):

    # Double the resolution in each direction
    ny = ny_initial * 2**i
    nx = round(ny * domain_length / domain_height)

    # Calculate the actual cell dimensions
    dy = domain_height / ny
    dx = domain_length / nx

    # Final element sizes (scaled with refinement)
    size_min = dy / 2
    size_max = dx * 2  # Allow larger elements away from walls
    refine_dist = domain_height / 4

    # Verify that the cells are square
    if not np.isclose(dx, dy):
        raise ValueError(
            f"Mesh {i + 1} does not have square cells: "
            f"dx={dx}, dy={dy}"
        )

    print(
        f"Mesh {i + 1}: "
        f"nx={nx}, ny={ny}, "
        f"dx={dx:.6e} m, dy={dy:.6e} m"
    )

    # Generate empty-channel mesh
    mesh_filename = generate_programmatic_hx(
        domain_height=domain_height,
        ny=ny,
        nx=nx,
        L_inlet=inlet_length,
        L_hx=hx_length,
        L_outlet=outlet_length,
        n_fins=None,
        fin_thickness=1,
        structured=False,
        size_min=size_min,
        size_max=size_max,
        refine_dist=refine_dist,
        refine_outer_walls=True,
        generate_mesh_file=True,
        ID=f"unstructured_ny{ny}",
    )

    print(f"Generated mesh file: {mesh_filename}")

    mesh_tests.append({
        "test": i + 1,
        "nx": nx,
        "ny": ny,
        "dx": dx,
        "dy": dy,
        "mesh_filename": mesh_filename,
    })

    input_filename = create_input_file(
        mesh_name=mesh_filename,
        template_path="baseline.i",
        output_dir=".",
        fluid_properties=fluid_properties,
        boundary_conditions=boundary_conditions,
    )

    print(f"Created MOOSE input file: {input_filename}")